# Bank Marketing - Feature Engineering and Preprocessing

This notebook rebuilds `Models/processed_bank_data.csv` from the raw UCI Bank Marketing zip using only relative project paths. It is intended to be re-runnable after cloning/copying the project.

Key corrections from the original draft:
- no Google Drive or machine-specific paths;
- exact duplicate rows are removed before encoding;
- target encoding and feature engineering are documented;
- `is_contacted_before` is created once in the processed dataset;
- output files are saved into the local `Models/` folder.


In [1]:
from __future__ import annotations

from pathlib import Path
import io
import json
import zipfile

import numpy as np
import pandas as pd


In [2]:
def find_project_root() -> Path:
    """Return the project root whether this notebook runs from root or from its own folder."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "EDA + Data" / "bank+marketing.zip").exists() and (candidate / "Models").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing EDA + Data/bank+marketing.zip and Models/")

PROJECT_ROOT = find_project_root()
RAW_ZIP = PROJECT_ROOT / "EDA + Data" / "bank+marketing.zip"
MODEL_DIR = PROJECT_ROOT / "Models"
OUTPUT_CSV = MODEL_DIR / "processed_bank_data.csv"
SUMMARY_JSON = MODEL_DIR / "preprocessing_summary.json"

print("Project root:", PROJECT_ROOT)
print("Raw zip:", RAW_ZIP)
print("Output CSV:", OUTPUT_CSV)


Project root: H:\Nam ba\Hoc ki 2\May Hoc\Project ML\Project after fixed
Raw zip: H:\Nam ba\Hoc ki 2\May Hoc\Project ML\Project after fixed\EDA + Data\bank+marketing.zip
Output CSV: H:\Nam ba\Hoc ki 2\May Hoc\Project ML\Project after fixed\Models\processed_bank_data.csv


In [3]:
def load_bank_additional_full(zip_path: Path) -> pd.DataFrame:
    """Load bank-additional-full.csv from UCI's nested bank+marketing.zip archive."""
    with zipfile.ZipFile(zip_path) as outer_zip:
        nested_zip_bytes = outer_zip.read("bank-additional.zip")

    with zipfile.ZipFile(io.BytesIO(nested_zip_bytes)) as inner_zip:
        with inner_zip.open("bank-additional/bank-additional-full.csv") as csv_file:
            return pd.read_csv(csv_file, sep=";")

raw_df = load_bank_additional_full(RAW_ZIP)
print("Raw shape:", raw_df.shape)
display(raw_df.head())


Raw shape: (41188, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [4]:
unknown_summary = (
    raw_df.select_dtypes(include="object")
    .eq("unknown")
    .sum()
    .rename("unknown_count")
    .to_frame()
)
unknown_summary["unknown_rate"] = unknown_summary["unknown_count"] / len(raw_df)
unknown_summary = unknown_summary[unknown_summary["unknown_count"] > 0].sort_values("unknown_rate", ascending=False)

quality_summary = pd.DataFrame(
    {
        "metric": ["rows", "columns", "exact_duplicates", "target_yes_rate", "pdays_999_rate"],
        "value": [
            len(raw_df),
            raw_df.shape[1],
            int(raw_df.duplicated().sum()),
            raw_df["y"].eq("yes").mean(),
            raw_df["pdays"].eq(999).mean(),
        ],
    }
)

display(quality_summary)
display(unknown_summary)


,metric,value
0,rows,41188.000000
1,columns,21.000000
2,exact_duplicates,12.000000
3,target_yes_rate,0.112654
4,pdays_999_rate,0.963217


,unknown_count,unknown_rate
default,8597,0.208726
education,1731,0.042027
housing,990,0.024036
loan,990,0.024036
job,330,0.008012
marital,80,0.001942


## Preprocessing Decisions

- Remove exact duplicate rows. There are only 12, but keeping them can slightly bias evaluation.
- Keep `unknown` as an explicit category instead of imputing it as missing. In this dataset it is an observed operational state.
- Encode `y`: `yes -> 1`, `no -> 0`.
- Add `age_group` for simple non-linear age structure.
- Add `is_contacted_before` because `pdays = 999` means the customer was not previously contacted.
- One-hot encode categorical variables with `drop_first=True` to avoid a redundant dummy column.
- Keep `duration` in the processed file, but drop it in realistic modeling scenarios because it is only known after the call.


In [5]:
def build_processed_data(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.drop_duplicates().reset_index(drop=True).copy()

    df["y"] = df["y"].map({"no": 0, "yes": 1}).astype(int)
    df["age_group"] = pd.cut(
        df["age"],
        bins=[0, 25, 40, 60, 120],
        labels=["young", "adult", "middle", "senior"],
        right=True,
    )
    df["is_contacted_before"] = (df["pdays"] != 999).astype(int)

    categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
    processed = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=bool)

    ordered_cols = [col for col in processed.columns if col != "y"] + ["y"]
    return processed[ordered_cols]

processed_df = build_processed_data(raw_df)
print("Processed shape:", processed_df.shape)
print("Target distribution:")
display(processed_df["y"].value_counts().rename_axis("y").reset_index(name="count"))
display(processed_df.head())


Processed shape: (41176, 58)
Target distribution:


,y,count
0,0,36537
1,1,4639


,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,...,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_nonexistent,poutcome_success,age_group_adult,age_group_middle,age_group_senior,y
0,56,261,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,True,False,False,False,True,False,False,True,False,0
1,57,149,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,True,False,False,False,True,False,False,True,False,0
2,37,226,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,True,False,False,False,True,False,True,False,False,0
3,40,151,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,True,False,False,False,True,False,True,False,False,0
4,56,307,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,True,False,False,False,True,False,False,True,False,0


In [6]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
processed_df.to_csv(OUTPUT_CSV, index=False)

summary = {
    "raw_shape": list(raw_df.shape),
    "duplicate_rows_removed": int(raw_df.duplicated().sum()),
    "processed_shape": list(processed_df.shape),
    "positive_rate": float(processed_df["y"].mean()),
    "target_counts": {str(k): int(v) for k, v in processed_df["y"].value_counts().sort_index().items()},
    "duration_policy": "Retained in processed data for benchmark only; dropped for realistic pre-call prediction.",
}
SUMMARY_JSON.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("Saved:", OUTPUT_CSV)
print("Saved:", SUMMARY_JSON)
summary


Saved: H:\Nam ba\Hoc ki 2\May Hoc\Project ML\Project after fixed\Models\processed_bank_data.csv
Saved: H:\Nam ba\Hoc ki 2\May Hoc\Project ML\Project after fixed\Models\preprocessing_summary.json


{'raw_shape': [41188, 21],
 'duplicate_rows_removed': 12,
 'processed_shape': [41176, 58],
 'positive_rate': 0.11266271614532737,
 'target_counts': {'0': 36537, '1': 4639},
 'duration_policy': 'Retained in processed data for benchmark only; dropped for realistic pre-call prediction.'}

## Output Contract

The modeling notebooks and Web Demo consume `Models/processed_bank_data.csv`. The file is already numeric/boolean, has no NaN values, and includes the target column `y`.
